# SynFS synthetic data example


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))


In [4]:


import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from src.trainer.trainer import SynFSTrainer
from src.data.dataset import SimpleDataset
from omegaconf import OmegaConf

from src.models.synfs_model import SynFSModel 

In [5]:
views_dims = [250,250]

In [6]:
cfg = OmegaConf.create({
    "model": {
        "views_dims": views_dims,
        "hidden_dims": [32,32],
        "output_dim":2,
        "batch_norm":True,
        "dropout":True,
        "activation": "relu",
        "learning_rate": 1e-3,
        "s_learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "s_lam": 0.1,
        "ns_lam": 1.07,
        "ns_alpha": 0.25,
    },
    "nr_epochs": 90,
    "seed": 0,
    "device": "cpu"
})

In [ ]:

from src.data.synfs_synthetic import (
    generate_multi_dataset,
    split_dataset,
)
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Generate EXACT SAME data as original notebook ----
views, y, (a_gt, s_gt) = generate_multi_dataset(
    n=20000,
    dims=views_dims,
    seed=0
)

# ---- Train/Val/Test split ----
(tr_X_set, tr_y), (va_X_set, va_y), (te_X_set, te_y) = split_dataset(views, y)

# ---- Wrap into PyTorch loaders ----
train_data = SimpleDataset(tr_X_set, tr_y, device)
val_data   = SimpleDataset(va_X_set, va_y, device)


train_loader = DataLoader(train_data, batch_size=250, drop_last=True, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=250, drop_last=False, shuffle=False)

print("Train size:", len(train_data))
print("Val size:", len(val_data))


validate: (array([0., 1.]), array([ 9876, 10124]))
Train size: 12800
Val size: 3200


In [9]:
xs, y = train_data[0]
len(xs), xs[0].shape, xs[1].shape # first sample dimension (n of views), 1st view dim, 2nd view dim


(2, torch.Size([250]), torch.Size([250]))

In [22]:
views

[array([[ 1.76405235,  0.40015721,  0.97873798, ...,  1.07961859,
         -0.81336426, -1.46642433],
        [ 0.52106488, -0.57578797,  0.14195316, ..., -0.51423397,
         -1.01804188, -0.07785476],
        [ 0.38273243, -0.03424228,  1.09634685, ...,  0.72003376,
         -1.82425666,  0.3036039 ],
        ...,
        [-0.28362113, -1.58276431, -0.75440427, ..., -0.82593719,
         -1.19462609, -0.21508922],
        [ 0.07367798,  1.11343913,  2.14680313, ..., -1.50289323,
         -0.21962545,  0.09748323],
        [-0.3594266 ,  1.30197362,  0.65692846, ...,  0.20278002,
          0.70692825,  0.73150149]]),
 array([[ 1.10815238,  0.58114489,  0.56379333, ..., -0.96474641,
          0.63741834, -0.45818475],
        [-0.55997876, -0.44168094, -0.58900462, ...,  1.0407    ,
         -0.23831184, -0.80338957],
        [-1.25962287, -0.47272067, -0.29010664, ..., -1.01459719,
         -0.61165017,  0.02894789],
        ...,
        [-0.12665823,  1.21621109,  0.05076089, ...,  

In [21]:
xs

[tensor([-2.2863e+00, -1.4002e-01,  1.9885e+00, -5.7425e-01,  1.2110e+00,
         -1.0476e+00, -2.2036e+00, -3.2983e-01, -3.4730e-01,  4.2538e-01,
         -7.3881e-02, -1.9414e+00,  1.2938e+00, -9.1864e-01,  9.3255e-01,
         -2.3035e-01, -4.8152e-01, -1.8093e-01, -2.5482e+00, -4.9725e-01,
         -1.0468e+00, -8.4236e-01,  1.1465e-02,  1.5822e-01,  1.1212e+00,
         -1.1012e+00, -1.5623e-01, -6.4259e-01, -4.7617e-01, -3.2178e-01,
          3.1430e-01,  3.2566e-02, -9.9988e-01, -5.4420e-01, -9.9821e-04,
          3.1217e-01,  1.2539e-01, -2.4448e+00,  5.7032e-01, -1.5207e+00,
          8.5147e-01,  3.1591e-02,  1.5469e+00,  1.8146e+00,  8.2597e-01,
          1.2194e-01, -5.6114e-01, -1.0149e-01,  7.1181e-02, -1.0993e-02,
         -6.0434e-01,  2.2601e-01,  9.6003e-01, -2.4163e-01, -3.2592e-01,
         -3.3457e-01, -4.7683e-01, -5.0096e-01, -3.2449e-01, -2.8297e-01,
         -1.0799e+00,  2.6982e-01,  2.7873e-01, -5.8749e-01, -1.1019e+00,
          5.7604e-01,  3.4965e-01,  1.

In [10]:
len(tr_X_set), tr_X_set[0].shape, tr_X_set[1].shape

(2, (12800, 250), (12800, 250))

In [12]:
from src.utils.feature_importance import get_important_features

In [13]:
model = SynFSModel(cfg.model).to(cfg.device)
trainer = SynFSTrainer(cfg,model)


history = {"train_loss": [], "val_auroc": []}
trainer.set_X_mean_set(train_loader)

for epoch in range(cfg.nr_epochs):
    train_metrics = trainer.train_epoch(train_loader)
    val_metrics   = trainer.validate_epoch(val_loader)

    history["train_loss"].append(train_metrics["loss"])
    history["val_auroc"].append(val_metrics["auroc"])


    if epoch % 30 == 0:
        print("======Epoch", epoch, "Train Loss:", train_metrics['loss'], "Val AUROC:", val_metrics['auroc'], "======")
        
        get_important_features(
            model,
            which="synergistic",
            threshold=0.7,
            verbose=True
        )
        get_important_features(
            model,
            which="non_synergistic",
            threshold=0.7,
            verbose=True
        )
        print('\n')

======Epoch 0 Train Loss: 1.4969176591611375 Val AUROC: 0.501571655312971 ======
[synergistic] threshold=0.7
  total features     : 500
  selected features  : 0
  indices: []
[non_synergistic] threshold=0.7
  total features     : 500
  selected features  : 0
  indices: []


======Epoch 30 Train Loss: 0.7269895065064523 Val AUROC: 0.7822725036433991 ======
[synergistic] threshold=0.7
  total features     : 500
  selected features  : 0
  indices: []
[non_synergistic] threshold=0.7
  total features     : 500
  selected features  : 2
  indices: [  2 253]


======Epoch 60 Train Loss: 0.6305221471132016 Val AUROC: 0.8115754318570704 ======
[synergistic] threshold=0.7
  total features     : 500
  selected features  : 2
  indices: [  0 251]
[non_synergistic] threshold=0.7
  total features     : 500
  selected features  : 2
  indices: [  2 253]




## Evaluation

### Feature Discovery

In [14]:
ns_gt = [ai - syn for ai, syn in zip(a_gt, s_gt)]
ns_gt = np.where(np.concatenate(ns_gt))[0]
s_gt = np.where(np.concatenate(s_gt))[0]
ground_truth_features = [ns_gt, s_gt]

In [15]:
ground_truth_features

[array([  2, 253]), array([  0, 251])]

In [18]:
from src.metric.metric import strict_jaccard,standard_metrics,tpr_fdr

In [19]:
# Get group similarity and group structure.
s = [gate.cpu().numpy() for gate in model.get_gates(model.s_model)]
n = [gate.cpu().numpy() for gate in model.get_gates(model.ns_model)]
n_predicted = np.where(np.concatenate(n)>0.7)[0]
s_predicted = np.where(np.concatenate(s)>0.7)[0]
predicted_features = [n_predicted, s_predicted]

# Get group similarity and group structure.
tpr, fdr = tpr_fdr(ground_truth_features, predicted_features)
j_index, ntrue, npredicted = strict_jaccard(ground_truth_features, predicted_features)

print("ground_truth_syn |", ground_truth_features[0], ",ground_truth_non-syn | ", ground_truth_features[1])
print("predicted_syn |", predicted_features[0], ",predicted_non-syn | ", predicted_features[1] )
print(
    "Jaccard Index: {:.3f}, True Positive Rate: {:.3f}%, False Discovery Rate: {:.3f}%".format(
        j_index, tpr, fdr
    )
)


ground_truth_syn | [  2 253] ,ground_truth_non-syn |  [  0 251]
predicted_syn | [  2 253] ,predicted_non-syn |  [  0 251]
Jaccard Index: 1.000, True Positive Rate: 100.000%, False Discovery Rate: 0.000%


### Predictive Performance

In [20]:
data = SimpleDataset(te_X_set, te_y, device=device)
testloader = DataLoader(data, batch_size=len(te_y))
           
res = []
for x, y in testloader:
    logits = trainer.predict(x)
    res.append(logits.detach().cpu().numpy())
logits = np.concatenate(res)

auroc, auprc, accuracy, f1 = standard_metrics(tr_y, te_y, logits, verbose=True)

auroc | 0.826, auprc | 0.825, accuracy | 0.749, f1 | 0.757
